# Cuando el camino más corto no es el mejor
## Simulación de evacuación en una zona inundada

**Objetivo:** comparar una ruta con pocos pasos con una ruta de menor costo
cuando el terreno presenta distintos grados de inundación. Ejecutar todas las
celdas en orden; no se necesitan archivos de datos externos.

**Modelo didáctico:** los niveles son categorías sintéticas, no profundidades
ni estimaciones reales de seguridad. Este prototipo no valida rutas de evacuación reales.

### Decisiones del modelo
- Una matriz rectangular representa el terreno: 0 = seco, 1 = inundación leve,
  2 = inundación moderada y 3 = inundación severa que bloquea el paso.
- El agente se mueve en cuatro direcciones. No hay diagonales ni saltos.
- Cada movimiento cuesta `1 + alfa * nivel_destino`. El punto inicial no se
  cobra porque no se entra en él. `alfa >= 0` expresa cuánto se penaliza el riesgo.
- El costo de una ruta es `pasos + alfa * exposición`, donde exposición es la
  suma de niveles de las celdas visitadas después del inicio.
- Es una suma ponderada: no promete minimizar primero el riesgo. Para dos rutas
  se calcula el costo con el mismo alfa y se elige el menor; un empate no implica
  que exista una única ruta óptima.

### Estrategias
**BFS** minimiza pasos, ignorando el riesgo al buscar; su costo ponderado se mide
después. **Dijkstra** minimiza el costo ponderado mediante relajación de distancias.
**A\*** usa el mismo costo y la distancia Manhattan al refugio como heurística.
Es admisible y consistente porque cada paso cuesta al menos 1; obstáculos y riesgo
solo pueden aumentar el costo. Por eso Dijkstra y A* deben obtener el mismo costo
óptimo, aunque sus rutas o trabajo pueden diferir. BFS solo es óptimo para el costo
cuando todos los pasos cuestan igual (por ejemplo, alfa = 0).

In [ ]:
from collections import deque
from dataclasses import dataclass
from heapq import heappop, heappush
from itertools import count
from math import inf, isfinite
import random

@dataclass
class Resultado:
    ruta: list | None
    expandidos: int

def validar_entrada(terreno, inicio, refugio, alfa):
    if not terreno or not terreno[0]:
        raise ValueError('El terreno no puede estar vacío.')
    ancho = len(terreno[0])
    if any(len(fila) != ancho for fila in terreno):
        raise ValueError('El terreno debe ser rectangular.')
    if any(type(nivel) is not int or nivel not in range(4)
           for fila in terreno for nivel in fila):
        raise ValueError('Los niveles deben ser enteros de 0 a 3.')
    if not isinstance(alfa, (int, float)) or not isfinite(alfa) or alfa < 0:
        raise ValueError('Alfa debe ser finito y no negativo.')
    for punto in (inicio, refugio):
        if (not isinstance(punto, tuple) or len(punto) != 2
            or any(type(x) is not int for x in punto)
            or not (0 <= punto[0] < len(terreno) and 0 <= punto[1] < ancho)):
            raise ValueError('Inicio o refugio fuera del terreno.')

def vecinos(terreno, punto):
    f, c = punto
    for df, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
        nf, nc = f + df, c + dc
        if 0 <= nf < len(terreno) and 0 <= nc < len(terreno[0]):
            if terreno[nf][nc] < 3:
                yield (nf, nc)

def reconstruir(padres, destino):
    ruta = []
    actual = destino
    while actual is not None:
        ruta.append(actual)
        actual = padres[actual]
    return ruta[::-1]

def buscar(terreno, inicio, refugio, estrategia='astar', alfa=2):
    validar_entrada(terreno, inicio, refugio, alfa)
    if estrategia not in ('bfs', 'dijkstra', 'astar'):
        raise ValueError('Estrategia desconocida.')
    if terreno[inicio[0]][inicio[1]] == 3 or terreno[refugio[0]][refugio[1]] == 3:
        return Resultado(None, 0)
    padres = {inicio: None}
    expandidos = 0
    if estrategia == 'bfs':
        frontera = deque([inicio])
        while frontera:
            actual = frontera.popleft()
            expandidos += 1
            if actual == refugio:
                return Resultado(reconstruir(padres, actual), expandidos)
            for siguiente in vecinos(terreno, actual):
                if siguiente not in padres:
                    padres[siguiente] = actual
                    frontera.append(siguiente)
    else:
        def h(p):
            return (abs(p[0] - refugio[0]) + abs(p[1] - refugio[1])
                    if estrategia == 'astar' else 0)
        orden = count()  # Desempate reproducible por orden de inserción.
        frontera = [(h(inicio), next(orden), 0, inicio)]
        distancias = {inicio: 0}
        while frontera:
            _, _, costo_actual, actual = heappop(frontera)
            if costo_actual != distancias[actual]:
                continue  # Descarta entradas obsoletas del heap.
            expandidos += 1
            if actual == refugio:
                return Resultado(reconstruir(padres, actual), expandidos)
            for siguiente in vecinos(terreno, actual):
                nuevo = costo_actual + 1 + alfa * terreno[siguiente[0]][siguiente[1]]
                if nuevo < distancias.get(siguiente, inf):
                    distancias[siguiente] = nuevo
                    padres[siguiente] = actual
                    heappush(frontera, (nuevo + h(siguiente), next(orden), nuevo, siguiente))
    return Resultado(None, expandidos)

def validar_ruta(terreno, inicio, refugio, ruta):
    # Comprobación independiente de vecinos(): detecta saltos y celdas bloqueadas.
    if not ruta or ruta[0] != inicio or ruta[-1] != refugio:
        return False
    for f, c in ruta:
        if not (0 <= f < len(terreno) and 0 <= c < len(terreno[0])):
            return False
        if terreno[f][c] == 3:
            return False
    return all(abs(a[0]-b[0]) + abs(a[1]-b[1]) == 1
               for a, b in zip(ruta, ruta[1:]))

def metricas(terreno, resultado, alfa):
    if resultado.ruta is None:
        return {'pasos': None, 'exposicion': None, 'costo': None,
                'expandidos': resultado.expandidos}
    pasos = len(resultado.ruta) - 1
    exposicion = sum(terreno[f][c] for f, c in resultado.ruta[1:])
    return {'pasos': pasos, 'exposicion': exposicion,
            'costo': pasos + alfa * exposicion, 'expandidos': resultado.expandidos}

## Escenarios controlados
El inicio está en (2, 0) y el refugio en (2, 6), con coordenadas (fila, columna).
El corredor central es directo pero inundado; las filas contiguas ofrecen desvíos secos.
La fila superior contiene inundación leve para mostrar los cuatro niveles del modelo.
Los cambios se hacen sobre copias para que una ejecución no contamine la siguiente.

In [ ]:
INICIO, REFUGIO = (2, 0), (2, 6)
ALFA = 2
BASE = [
    [0, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 2, 2, 2, 2, 2, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [3, 3, 0, 0, 0, 3, 3],
]
def modificar(terreno, cambios):
    copia = [fila[:] for fila in terreno]
    for (f, c), nivel in cambios.items():
        copia[f][c] = nivel
    return copia

ruta_previa = buscar(BASE, INICIO, REFUGIO, 'astar', ALFA).ruta
CELDA_BLOQUEADA = ruta_previa[len(ruta_previa)//2]
ESCENARIOS = {
    'Base: ruta directa con riesgo': BASE,
    'Bloqueo de una celda de la ruta': modificar(BASE, {CELDA_BLOQUEADA: 3}),
    'Refugio aislado': modificar(BASE, {(1, 6): 3, (2, 5): 3, (3, 6): 3}),
    'Corredor central ahora seco': modificar(BASE, {(2, c): 0 for c in range(1, 6)}),
}
ESTRATEGIAS = ('bfs', 'dijkstra', 'astar')
resultados = {}
filas = []
print(f"{'Escenario':38} {'Algoritmo':9} {'Pasos':>6} {'Riesgo':>6} {'Costo':>6} {'Expandidos':>10}")
for nombre, terreno in ESCENARIOS.items():
    resultados[nombre] = {}
    for estrategia in ESTRATEGIAS:
        resultado = buscar(terreno, INICIO, REFUGIO, estrategia, ALFA)
        resultados[nombre][estrategia] = resultado
        m = metricas(terreno, resultado, ALFA)
        filas.append({'escenario': nombre, 'algoritmo': estrategia, **m})
        valores = [str(m[k]) if m[k] is not None else '—'
                   for k in ('pasos', 'exposicion', 'costo', 'expandidos')]
        print(f'{nombre:38} {estrategia:9} {valores[0]:>6} {valores[1]:>6} {valores[2]:>6} {valores[3]:>10}')